Notebook for results

In [1]:
#Import libraries

#import libraries

import scicone
import numpy as np
import pickle
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import io
import scanpy as sc
import anndata
import pyranges
import gseapy as gp
import json

# Set up SCICoNE
install_path = '/cluster/work/bewi/members/andress/pylabs/SCICoNE/build/'
install_path_local = "/home/andress/pylabs/SCICoNE_lab/build/"
temporary_outpath = './'
adatas_path = '/home/andress/pylabs/SCICoNE_lab/rna_imp/adatas'

seed = 42 # for reproducibility

np.random.seed(seed)

# Create SCICoNE object
sci = scicone.SCICoNE(install_path_local, temporary_outpath, verbose=False)

Using binaries at /home/andress/pylabs/SCICoNE_lab/build/


In [ ]:
gr_annotations_df = pd.read_csv('/home/andress/pylabs/SCICoNE_lab/rna_imp/gr_annotations.csv')

In [ ]:
adata = anndata.read_h5ad('/home/andress/pylabs/SCICoNE_lab/rna_imp/adatas/adata_leiden.h5ad') 

gr_annotations = pd.read_csv('/home/andress/pylabs/SCICoNE_lab/rna_imp/gr_annotations.csv')

df_annotations = gr_annotations.set_index('ensembl_gene_id')

df_exp_annotations = df_annotations.loc[df_annotations.index.intersection(adata.var['ensembl_gene_id'])]\
                        .reset_index().rename(columns={'index':'ensembl_gene_id'})

adata.var_names = adata.var['ensembl_gene_id'].values

adata = adata[:,df_exp_annotations['ensembl_gene_id']]


df_exp_annotations_sorted = df_exp_annotations.sort_values(by=['Chromosome', 'End'])


In [ ]:
#SORT ADATA OBJECT (already sorted)

# Merge Chromosome and End information into adata.var
# adata.var = adata.var.merge(gr_annotations_df[['ensembl_gene_id', 'Chromosome', 'End']], 
#                             on='ensembl_gene_id', how='left')

# print(adata.var.columns)

# adata.var['Chromosome'] = adata.var['Chromosome'].astype(str)  # Convert Chromosome to string

# # Sort adata.var by Chromosome and End
# sorted_var = adata.var.sort_values(by=['Chromosome', 'End'])

# # Reorder adata.X columns based on the sorted var index
# adata = adata[:, sorted_var.index]

In [ ]:
chr_var_names = dict()
for chromosome in df_exp_annotations_sorted['Chromosome'].unique():
    chr_var_names[chromosome] = df_exp_annotations_sorted.query(f' Chromosome=="{chromosome}" ')\
                                .sort_values('Start')['ensembl_gene_id'].values

# Load the JSON file as a dictionary
json_file_path = '/home/andress/pylabs/SCICoNE_lab/rna_imp/chromosome_stops.json'
with open(json_file_path, 'r') as file:
    chromosome_stops = json.load(file)

#sort the dictionary by values
chromosome_stops = {k: v for k, v in sorted(chromosome_stops.items(), key=lambda item: item[1])}
chromosome_stops = {np.str_(k): np.int64(v) for k, v in chromosome_stops.items()}

# Use chromosome_stops directly without further processing
#add 0 and final position
chromosome_stops_list = [np.int64(0)] + sorted(list(chromosome_stops.values())) + [np.int64(adata.shape[1]-1)]

print(chromosome_stops_list)


In [ ]:
file_list = ['clusters_median', 'clusters_mean', 'clusters_sum', 'clusters_mean_transformed',
             'clusters_sum_transformed', 'clusters_median_transformed', 'adata_clustered_norm']
for cluster_file in file_list:
    #for window_size in range(100, 1000, 100):
            try:
                adata = anndata.read_h5ad(f'{adatas_path}/{cluster_file}.h5ad')
                data = adata.X
                # Run SCICoNE analysis
                bps = sci.detect_breakpoints(data=data, window_size=200, threshold=3, input_breakpoints=chromosome_stops_list)

                # Save the matrix plot
                matrix_plot_filename = f"{temporary_outpath}/matrix_plot_{cluster_file}.png"
                scicone.plotting.plot_matrix(data, bps = bps['segmented_regions'],
                                                chr_stops_dict=chromosome_stops,
                                                cbar_title='Normalized\n  counts', vmax=2, cluster=False)

                # plt.savefig(matrix_plot_filename, dpi=300, bbox_inches='tight')
                # plt.close()

                # Learn and save the tree plot
                
                inferred_tree = sci.learn_tree(data, bps["segmented_region_sizes"], n_reps = 4, seed = seed, max_tries = 1)
                inferred_tree.plot_tree(gene_labels=True, node_labels=True, node_sizes=True, event_fontsize=8, nodesize_fontsize=10)

                tree_plot_filename = f"{temporary_outpath}/tree_plot_{cluster_file}.png"
                # Render the tree plot using graphviz's render method

                # inferred_tree.render(tree_plot_filename, format='png', dpi = 600, cleanup=True)
                # print(f"Tree plot saved to {tree_plot_filename}.png")

            except ValueError as ve:
                print(f"ValueError while processing the cluster file: {ve}")
            except Exception as e:
                print(f"An error occurred while processing the cluster file: {e}")
                continue

In [ ]:
cluster_file_1 = 'clusters_mean'
cluster_file_2 = 'clusters_median'
cluster_file_3 = 'clusters_sum'
cluster_file_4 = 'clusters_mean_transformed'
cluster_file_5 = 'clusters_sum_transformed'
cluster_file_6 = 'clusters_median_transformed'
cluster_file_7 = 'adata_clustered_norm'
cluster_file_8 = 'adata_clusters_sum'

In [ ]:
for cluster_file in [cluster_file_1, cluster_file_2,cluster_file_3, cluster_file_4,
                     cluster_file_5, cluster_file_6, cluster_file_7]:
    try:
        adata = anndata.read_h5ad(f'{adatas_path}/{cluster_file}.h5ad')
        data = adata.X
        # Run SCICoNE analysis
        bps = sci.detect_breakpoints(data=data, window_size=200, threshold=3, input_breakpoints=chromosome_stops_list)

        # Save the matrix plot
        matrix_plot_filename = f"{temporary_outpath}/matrix_plot_{cluster_file}.png"
        
        scicone.plotting.plot_matrix(data, bps = bps['segmented_regions'],
                                        chr_stops_dict=chromosome_stops,
                                        cbar_title='Normalized\n  counts', vmax=2, cluster=False)

        # plt.savefig(matrix_plot_filename, dpi=300, bbox_inches='tight')
        # plt.close()

        # Learn and save the tree plot
        
        # inferred_tree = sci.learn_tree(data, bps["segmented_region_sizes"], n_reps = 4, seed = seed, max_tries = 1)
        # inferred_tree.plot_tree(gene_labels=True, node_labels=True, node_sizes=True, event_fontsize=8, nodesize_fontsize=10)

        # tree_plot_filename = f"{temporary_outpath}/tree_plot_{cluster_file}.png"
        # Render the tree plot using graphviz's render method

        # inferred_tree.render(tree_plot_filename, format='png', dpi = 600, cleanup=True)
        # print(f"Tree plot saved to {tree_plot_filename}.png")

    except ValueError as ve:
        print(f"ValueError while processing the cluster file: {ve}")
    except Exception as e:
        print(f"An error occurred while processing the cluster file: {e}")
        continue


In [ ]:
cluster_file = 'adata_clusters_mean'
adata = anndata.read_h5ad(f'{adatas_path}/{cluster_file}.h5ad')
data = adata.X
# Run SCICoNE analysis
bps = sci.detect_breakpoints(data=data, window_size=200, threshold=3, input_breakpoints=chromosome_stops_list)

        # Save the matrix plot
matrix_plot_filename = f"{temporary_outpath}/matrix_plot_{cluster_file}.png"
scicone.plotting.plot_matrix(data, bps = bps['segmented_regions'],
                                chr_stops_dict=chromosome_stops,
                                cbar_title='Normalized\n  counts', vmax=2, cluster=False)
